# Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
### Module 2: Phase Space and Quasiprobability Distributions
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/02_wigner_negativity_and_nonclassicality.ipynb)

---

### Prerequisites
- Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
- Notebook 1.1: Fock States, Ladder Operators, and Truncation Artifacts
- Notebook 1.3: Squeezed Vacuum and Quadrature Noise Reduction

### Pedagogical Learning Objectives
1. **Hudson's Theorem (1974):** Understand and prove why the only pure quantum states with strictly non-negative Wigner functions ($W(x, p) \ge 0$) are Gaussian states (displaced squeezed vacua).
2. **Quantifying Non-Classicality via Negativity Volume:** Formulate the Kenfack-Życzkowski Wigner negativity metric $\delta(\hat{\rho}) = \iint (|W| - W) \, dx \, dp = 2 \iint_{W<0} |W| \, dx \, dp$.
3. **Fock State Negativity Scaling:** Derive the exact analytical negativity of the single-photon state $\delta(|1\rangle) = 4e^{-1/2} - 2 \approx 0.4261$, and analyze the monotonic growth of $\delta(|n\rangle)$ with photon number $n$.
4. **Superposition and Symmetry Breaking:** Investigate how quantum superpositions ($c_0|0\rangle + c_1|1\rangle$) break phase-space rotational symmetry and morph negative regions into interference crescents.
5. **Continuous-Variable Computational Advantage:** Connect Wigner negativity to the Gottesman-Knill theorem, proving why $W < 0$ is an indispensable resource for universal quantum computation.
6. **Numerical Benchmarking:** Implement fast 2D quadrature integration routines to compute $\delta(\hat{\rho})$ and verify invariance under displacement and rotation.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Hudson's Theorem (1974) and the Gaussian Boundary

In 1974, mathematician R. L. Hudson established one of the most celebrated theorems in modern quantum theory (*Reports on Mathematical Physics* 6, 249, 1974):

> **Hudson's Theorem:**
> A pure quantum state $|\psi\rangle$ possesses an everywhere non-negative Wigner quasiprobability distribution:
> $$
> W(x, p) \ge 0 \quad \text{for all } (x, p) \in \mathbb{R}^2
> $$
> **if and only if** $|\psi\rangle$ is a **Gaussian state**!

---

### What is a Pure Gaussian State?

A pure state is Gaussian if its coordinate wavefunction is the exponential of a quadratic polynomial:
$$
\psi_{\text{Gauss}}(x) = \left( \frac{a}{\pi} \right)^{1/4} \exp\left( -\frac{a + ib}{2} (x - x_0)^2 + i p_0 x \right)
$$
In quantum optics, **every pure Gaussian state is a displaced squeezed vacuum state**:
$$
|\psi_{\text{Gauss}}\rangle = \hat{D}(\alpha) \hat{S}(\xi) |0\rangle
$$
This family includes:
1. The **vacuum state** $|0\rangle$ ($\alpha = 0, \xi = 0$).
2. All **coherent states** $|\alpha\rangle$ ($\xi = 0$).
3. All **squeezed vacua** $|0, \xi\rangle$ ($\alpha = 0$).
4. All **displaced squeezed states** $|\alpha, \xi\rangle$.

---

### The Profound Physical Consequence

If a pure state is **non-Gaussian** (for example, any Fock state $|n\rangle$ with $n \ge 1$, Schrödinger cat states, or GKP quantum error-correcting code states), **its Wigner function MUST take negative values somewhere in phase space!**

Wigner negativity is not a mathematical defect or pathology: it is the **defining geometric signature of non-Gaussian quantum interference**.


## 2. Quantifying Non-Classicality: The Wigner Negativity Volume $\delta(\hat{\rho})$

While observing $W(x, p) < 0$ at a single point signals non-classicality, how can we quantify *how much* non-classicality a state possesses?

In 2004, **A. Kenfack and K. Życzkowski** introduced a rigorous, continuous measure (*J. Opt. B: Quantum Semiclass. Opt.* 6, 396–404, 2004): the **Wigner Negativity Volume** $\delta(\hat{\rho})$.

---

### Mathematical Definition

$$
\delta(\hat{\rho}) \equiv \iint_{-\infty}^\infty \left[ |W(x, p)| - W(x, p) \right] \, dx \, dp
$$

Using the unit normalization condition $\iint W(x, p) \, dx \, dp = 1$, the integral decomposes into positive and negative domains:
$$
\iint |W(x, p)| \, dx \, dp = \iint_{W \ge 0} W(x, p) \, dx \, dp - \iint_{W < 0} W(x, p) \, dx \, dp
$$
$$
1 = \iint_{W \ge 0} W(x, p) \, dx \, dp + \iint_{W < 0} W(x, p) \, dx \, dp
$$
Subtracting the two equations yields:
$$
\iint |W(x, p)| \, dx \, dp - 1 = -2 \iint_{W < 0} W(x, p) \, dx \, dp = 2 \iint_{W < 0} |W(x, p)| \, dx \, dp
$$

Therefore:
$$
\delta(\hat{\rho}) = \iint_{-\infty}^\infty |W(x, p)| \, dx \, dp - 1 = 2 \iint_{W < 0} |W(x, p)| \, dx \, dp
$$

---

### Key Properties of $\delta(\hat{\rho})$:
1. **Faithfulness:** $\delta(\hat{\rho}) \ge 0$. For any Gaussian state (pure or mixed), $\delta(\hat{\rho}) = 0$. For any non-Gaussian pure state, $\delta(\hat{\rho}) > 0$.
2. **Symplectic Invariance:** $\delta(\hat{\rho})$ is strictly invariant under all Gaussian unitary operations, including phase-space displacements $\hat{D}(\alpha)$ and phase rotations $e^{-i\theta \hat{n}}$:
   $$
   \delta\left( \hat{D}(\alpha)\hat{\rho}\hat{D}^\dagger(\alpha) \right) = \delta(\hat{\rho})
   $$
3. **Monotonic Ordering:** For Fock states, $\delta(|n\rangle)$ grows monotonically with $n$, serving as an operational measure of non-classical resource content!


In [ ]:
# Implementation of General Wigner Negativity Volume Solver

def compute_wigner_grid(state, xvec, pvec):
    """Computes W(x, p) on a 2D Cartesian grid using Fock-basis Laguerre expansion."""
    if state.ndim == 1:
        psi = state / np.linalg.norm(state)
        rho = np.outer(psi, psi.conj())
    else:
        rho = state / np.real(np.trace(state))
        
    N = rho.shape[0]
    X, P = np.meshgrid(xvec, pvec)
    alpha = (X + 1j * P) / np.sqrt(2.0)
    z = 4.0 * np.abs(alpha)**2
    exp_factor = (1.0 / np.pi) * np.exp(-2.0 * np.abs(alpha)**2)
    
    W = np.zeros_like(X, dtype=np.complex128)
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    
    for n in range(N):
        if abs(rho[n, n]) > 1e-15:
            L_nn = genlaguerre(n, 0)(z)
            W += rho[n, n] * ((-1)**n) * L_nn
            
        for m in range(n + 1, N):
            if abs(rho[n, m]) > 1e-15:
                d = m - n
                coeff = ((-1)**n) * np.sqrt(fact[n] / fact[m]) * ((2.0 * np.conj(alpha))**d)
                L_nm = genlaguerre(n, d)(z)
                term = coeff * L_nm
                W += rho[n, m] * term + rho[m, n] * np.conj(term)
                
    return np.real(W) * exp_factor

def wigner_negativity_volume(W, xvec, pvec):
    """Computes Kenfack-Zyczkowski negativity volume delta = 2 * int_{W<0} |W| dx dp."""
    dx = xvec[1] - xvec[0]
    dp = pvec[1] - pvec[0]
    dA = dx * dp
    
    neg_mask = W < 0.0
    neg_volume = 2.0 * np.sum(np.abs(W[neg_mask])) * dA
    total_abs_integral = np.sum(np.abs(W)) * dA
    
    return neg_volume, total_abs_integral

# Benchmark: Analytical verification for single-photon state |1>
# Analytical formula: delta(|1>) = 4 * exp(-0.5) - 2 = 0.4261226...
analytical_delta_1 = 4.0 * np.exp(-0.5) - 2.0

N_dim = 15
fock1 = np.zeros(N_dim, dtype=np.complex128)
fock1[1] = 1.0

x_eval = np.linspace(-4.5, 4.5, 121)
p_eval = np.linspace(-4.5, 4.5, 121)

W_1_grid = compute_wigner_grid(fock1, x_eval, p_eval)
delta_num, total_abs = wigner_negativity_volume(W_1_grid, x_eval, p_eval)

print(f"--- Single-Photon Fock State |1> Negativity Verification ---")
print(f"Numerical Negativity delta:  {delta_num:.6f}")
print(f"Analytical Exact delta:      {analytical_delta_1:.6f}")
print(f"Absolute Error:              {abs(delta_num - analytical_delta_1):.2e}")
print(f"Total Integral of |W|:       {total_abs:.6f} (Expected: 1 + delta = {1.0 + analytical_delta_1:.6f})")

assert abs(delta_num - analytical_delta_1) < 1e-3, "Fock |1> negativity test failed!"
print("Analytical and numerical Wigner negativity volume successfully benchmarked!")


## 3. Fock State Negativity Scaling and Zero-Crossing Radii

Recall the analytical formula for the Wigner function of an $n$-photon Fock state:

$$
W_n(r) = \frac{(-1)^n}{\pi} e^{-r^2} L_n(2r^2), \qquad r^2 = x^2 + p^2
$$

Because $W_n(r)$ is purely a function of radial distance $r$, its nodal zero-crossings ($W_n(r) = 0$) correspond to concentric circles in phase space:

$$
L_n(2r_0^2) = 0
$$

---

### Zero-Crossing Contour Radii:
- **$n = 0$ (Vacuum):** $L_0(u) = 1 \neq 0$ (No zeros; everywhere positive).
- **$n = 1$ (Single Photon):** $L_1(u) = 1 - u = 0 \implies u = 1 \implies 2r_0^2 = 1$:
  $$
  r_0 = \frac{1}{\sqrt{2}} \approx 0.7071
  $$
  For $r < r_0$, $W_1(r) < 0$ (central negative core). For $r > r_0$, $W_1(r) > 0$ (outer positive ring).
- **$n = 2$:** $L_2(u) = 1 - 2u + \frac{1}{2}u^2 = 0 \implies u = 2 \pm \sqrt{2}$:
  $$
  r_{0, 1} = \sqrt{1 - 1/\sqrt{2}} \approx 0.541, \qquad r_{0, 2} = \sqrt{1 + 1/\sqrt{2}} \approx 1.307
  $$
  The state oscillates between positive core, negative annular ring, and positive outer rim!

As $n$ increases, the number of alternating rings grows as $n$, and the total negativity volume $\delta(|n\rangle)$ scales upward monotonically.


In [ ]:
# Scaling Analysis: Wigner Negativity Volume delta across Fock States n = 0 to 6

n_max = 6
fock_deltas = []
r_fine = np.linspace(0, 6.0, 5000)
dr_fine = r_fine[1] - r_fine[0]

# High-precision 1D polar integration for radial states:
# delta = 2 * int_{W<0} |W(r)| * 2*pi*r dr
for n_idx in range(n_max + 1):
    u = 2.0 * (r_fine**2)
    w_r = ((-1.0)**n_idx / np.pi) * np.exp(-r_fine**2) * genlaguerre(n_idx, 0)(u)
    neg_mask = w_r < 0.0
    neg_vol_polar = 2.0 * np.sum(np.abs(w_r[neg_mask]) * 2.0 * np.pi * r_fine[neg_mask] * dr_fine)
    fock_deltas.append(neg_vol_polar)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Radial Profiles W_n(r)
colors_fock = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728', '#9467bd', '#8c564b', '#e377c2']
for n_idx in range(4):
    u = 2.0 * (r_fine**2)
    w_r = ((-1.0)**n_idx / np.pi) * np.exp(-r_fine**2) * genlaguerre(n_idx, 0)(u)
    ax1.plot(r_fine, w_r, color=colors_fock[n_idx], lw=2.2, label=rf'Fock $|{n_idx}\rangle$')

ax1.axhline(0.0, color='black', lw=1.2, ls='--')
ax1.set_xlim(0, 3.5)
ax1.set_ylim(-0.35, 0.35)
ax1.set_xlabel(r"Phase-Space Radius $r = \sqrt{x^2 + p^2}$")
ax1.set_ylabel(r"Wigner Amplitude $W(r)$")
ax1.set_title("Radial Wigner Profiles: Concentric Quantum Rings")
ax1.legend(loc='upper right', framealpha=0.9)
ax1.grid(True)

# Panel 2: Negativity Volume delta(|n>) vs Photon Number n
n_axis = np.arange(n_max + 1)
bars = ax2.bar(n_axis, fock_deltas, color='#1f77b4', alpha=0.8, edgecolor='black', width=0.55)
ax2.plot(n_axis, fock_deltas, 'ro-', lw=2, markersize=6)

for bar, d_val in zip(bars, fock_deltas):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f"{d_val:.3f}", ha='center', va='bottom', fontweight='bold', fontsize=9)

ax2.set_xlim(-0.6, n_max + 0.6)
ax2.set_ylim(0, max(fock_deltas) * 1.25)
ax2.set_xlabel(r"Fock Number $n$")
ax2.set_ylabel(r"Wigner Negativity Volume $\delta(|n\rangle)$")
ax2.set_title(r"Monotonic Growth of Wigner Negativity: $\delta(|n\rangle)$")
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()

for n_val, d_val in zip(n_axis, fock_deltas):
    print(f"Fock state |{n_val}>: delta = {d_val:.4f}")


In [ ]:
# Phase-Space Symmetry Breaking in Superposition States: c0|0> + c1|1>

# Construct superposition |psi> = cos(theta/2) |0> + sin(theta/2) exp(i*phi) |1>
theta_sup = np.pi / 2.0  # Equal weights: 1/sqrt(2) (|0> + |1>)
phi_sup = 0.0

c0 = np.cos(theta_sup / 2.0)
c1 = np.sin(theta_sup / 2.0) * np.exp(1j * phi_sup)

psi_super = np.zeros(N_dim, dtype=np.complex128)
psi_super[0] = c0
psi_super[1] = c1

W_super = compute_wigner_grid(psi_super, x_eval, p_eval)
delta_super, _ = wigner_negativity_volume(W_super, x_eval, p_eval)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel 1: 2D Contour showing crescent negativity
levels_sup = np.linspace(-0.25, 0.35, 45)
cf = ax1.contourf(x_eval, p_eval, W_super, levels=levels_sup, cmap='RdBu_r', extend='both')
# Zero contour
ax1.contour(x_eval, p_eval, W_super, levels=[0.0], colors='black', linewidths=2.0, linestyles='--')

ax1.set_aspect('equal')
ax1.set_xlim(-3.5, 3.5)
ax1.set_ylim(-3.5, 3.5)
ax1.set_xlabel(r"Quadrature $x$")
ax1.set_ylabel(r"Quadrature $p$")
ax1.set_title(rf"Superposition $\frac{{1}}{{\sqrt{{2}}}}(|0\rangle + |1\rangle)$: $\delta={delta_super:.3f}$")
plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04, label=r'$W(x, p)$')

# Panel 2: Slice along x (p = 0) showing asymmetric dip
center_p_idx = len(p_eval) // 2
slice_x = W_super[center_p_idx, :]
ax2.plot(x_eval, slice_x, 'b-', lw=2.5, label=r'$W(x, p=0)$')
ax2.axhline(0.0, color='black', lw=1.2, ls='--')
ax2.fill_between(x_eval, slice_x, 0, where=(slice_x < 0), color='red', alpha=0.35, label='Negative Region ($W < 0$)')

ax2.set_xlim(-3.5, 3.5)
ax2.set_xlabel(r"Quadrature $x$")
ax2.set_ylabel(r"Wigner Amplitude $W(x, 0)$")
ax2.set_title("Cross-Section along x: Broken Reflection Symmetry")
ax2.legend(loc='upper right')
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Superposition state (|0> + |1>)/sqrt(2) Negativity Volume delta: {delta_super:.4f}")


> **Physical Insight: Wigner Negativity as a Continuous-Variable Computational Resource**
> In discrete-variable quantum computing (qubits), the Gottesman-Knill theorem proves that quantum circuits consisting purely of Clifford gates can be efficiently simulated by a classical Turing machine. Magic states with non-positive discrete Wigner functions are required to achieve universal quantum speedup.
>
> In **continuous-variable (CV) quantum computing**, an exact parallel exists:
> - **Gaussian states + Gaussian operations** (beam splitters, phase shifters, squeezers, and homodyne measurements) are **strictly classically simulable in polynomial time** (Mari & Eisert, *Phys. Rev. Lett.* 109, 230503, 2012).
> - **Wigner Negativity is a strictly necessary resource** for any continuous-variable quantum algorithm to demonstrate quantum supremacy over classical supercomputers!


In [ ]:
# Interactive Exploration: Real-Time Wigner Negativity Tuner

def explore_superposition_negativity(weight_1=0.5, phase_deg=0.0):
    """Interactive exploration of Wigner negativity in c0|0> + c1 exp(i phi)|1>."""
    c1_amp = np.sqrt(weight_1)
    c0_amp = np.sqrt(max(0.0, 1.0 - weight_1))
    phi_rad = np.radians(phase_deg)
    
    N_calc = 15
    psi = np.zeros(N_calc, dtype=np.complex128)
    psi[0] = c0_amp
    psi[1] = c1_amp * np.exp(1j * phi_rad)
    
    x_interactive = np.linspace(-3.5, 3.5, 61)
    p_interactive = np.linspace(-3.5, 3.5, 61)
    
    W = compute_wigner_grid(psi, x_interactive, p_interactive)
    delta_val, _ = wigner_negativity_volume(W, x_interactive, p_interactive)
    min_w = np.min(W)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. 2D Contour
    v_abs = max(0.35, np.max(np.abs(W)))
    cf = ax1.contourf(x_interactive, p_interactive, W, levels=np.linspace(-v_abs, v_abs, 40),
                      cmap='RdBu_r', extend='both')
    ax1.contour(x_interactive, p_interactive, W, levels=[0.0], colors='black', linewidths=1.5, linestyles='--')
    ax1.set_aspect('equal')
    ax1.set_xlabel("x")
    ax1.set_ylabel("p")
    ax1.set_title(rf"State: $\sqrt{{{1-weight_1:.2f}}}|0\rangle + \sqrt{{{weight_1:.2f}}}e^{{i{phase_deg:.0f}^\circ}}|1\rangle$")
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)
    
    # 2. Negativity Volume Gauge
    ax2.barh([0], [delta_val], color='#d62728' if delta_val > 0.01 else '#2ca02c', height=0.4, edgecolor='black')
    ax2.axvline(analytical_delta_1, color='blue', ls=':', lw=2, label=rf'Fock $|1\rangle$ Bound ({analytical_delta_1:.3f})')
    ax2.set_xlim(0, 0.55)
    ax2.set_yticks([])
    ax2.set_xlabel(r"Wigner Negativity Volume $\delta(\hat{\rho})$")
    ax2.set_title(f"Negativity Volume: delta = {delta_val:.4f} (Min W: {min_w:+.3f})")
    ax2.legend(loc='upper right')
    ax2.grid(True, axis='x')
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_superposition_negativity,
             weight_1=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5, description='Weight |1>:'),
             phase_deg=FloatSlider(min=0.0, max=360.0, step=15.0, value=0.0, description='Phase (deg):'))
else:
    # Static execution fallback for headless environments
    explore_superposition_negativity(weight_1=0.5, phase_deg=0.0)


## 4. Self-Check Exercises and Pedagogical Challenges

Deepen your understanding of Hudson's theorem and phase-space non-classicality measures.

---

### Challenge 1: Analytical Derivation of the Zero-Crossing Radius for Fock State $|1\rangle$
1. Using the formula $W_1(r) = \frac{1}{\pi} e^{-r^2} (2r^2 - 1)$, find the exact radius $r_0$ of the boundary separating the negative core from the positive outer ring.
2. Evaluate the integral $\delta(|1\rangle) = 2 \iint_{W < 0} |W_1(x, p)| \, dx \, dp$ in polar coordinates to prove that $\delta(|1\rangle) = 4e^{-1/2} - 2$.

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The boundary where the Wigner function vanishes satisfies $W_1(r_0) = 0$:
$$
2r_0^2 - 1 = 0 \quad \Longrightarrow \quad r_0^2 = \frac{1}{2} \quad \Longrightarrow \quad r_0 = \frac{1}{\sqrt{2}} \approx 0.7071
$$
For $r < 1/\sqrt{2}$, $2r^2 - 1 < 0 \implies W_1(r) < 0$.

**Part 2 Solution:**
In polar coordinates, $dx \, dp = r \, dr \, d\theta = 2\pi r \, dr$.
The negativity volume is:
$$
\delta(|1\rangle) = 2 \int_0^{1/\sqrt{2}} 2\pi r \, dr \cdot \left| \frac{1}{\pi} e^{-r^2} (2r^2 - 1) \right| = 4 \int_0^{1/\sqrt{2}} r e^{-r^2} (1 - 2r^2) \, dr
$$
Make substitution $u = r^2$, $du = 2r \, dr$. The upper limit is $r_0^2 = 1/2$:
$$
\delta(|1\rangle) = 2 \int_0^{1/2} e^{-u} (1 - 2u) \, du
$$
Integrate each term:
$$
\int_0^{1/2} e^{-u} \, du = 1 - e^{-1/2}
$$
$$
\int_0^{1/2} 2u e^{-u} \, du = \left[ -2u e^{-u} \right]_0^{1/2} + 2 \int_0^{1/2} e^{-u} \, du = -e^{-1/2} + 2(1 - e^{-1/2}) = 2 - 3e^{-1/2}
$$
Combining terms:
$$
\delta(|1\rangle) = 2 \left[ (1 - e^{-1/2}) - (2 - 3e^{-1/2}) \right] = 2 (2e^{-1/2} - 1) = 4e^{-1/2} - 2 \approx 0.4261226 \quad \blacksquare
$$
</details>

---

### Challenge 2: Invariance of $\delta(\hat{\rho})$ Under Displacement
Using QuTiP or NumPy, compute the negativity volume $\delta$ of a displaced single-photon state $|\psi\rangle = \hat{D}(\alpha)|1\rangle$ for $\alpha = 1.5$, and verify that $\delta$ equals that of the undisplaced $|1\rangle$ state within numerical grid tolerance.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

```python
# NumPy Verification of Displacement Invariance
N = 25
alpha_disp = 1.5
a = np.diag(np.sqrt(np.arange(1, N)), k=1)
D_op = la.expm(alpha_disp * a.T.conj() - alpha_disp * a)

fock1_state = np.zeros(N, dtype=np.complex128); fock1_state[1] = 1.0
disp_fock1 = D_op @ fock1_state

x_test = np.linspace(-5, 5, 121)
p_test = np.linspace(-5, 5, 121)

W_disp = compute_wigner_grid(disp_fock1, x_test, p_test)
delta_disp, _ = wigner_negativity_volume(W_disp, x_test, p_test)

print(f"Undisplaced |1> delta: {analytical_delta_1:.6f}")
print(f"Displaced D(1.5)|1> delta: {delta_disp:.6f}")
print(f"Difference: {abs(delta_disp - analytical_delta_1):.2e}")
assert abs(delta_disp - analytical_delta_1) < 5e-3, "Displacement invariance failed!"
print("Displacement invariance verified!")
```
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Non-Classicality Metrics

| Metric | Definition | Positive for Classical Light? | Invariant under Displacement? | Computational Resource Witness? |
| :--- | :--- | :---: | :---: | :---: |
| **Mandel $Q$** | $\frac{\text{Var}(n) - \langle n \rangle}{\langle n \rangle}$ | Yes ($Q \ge 0$) | No | Sub-Poissonian witness |
| **Coherence $g^{(2)}(0)$** | $\frac{\langle (\hat{a}^\dagger)^2 \hat{a}^2 \rangle}{\langle \hat{a}^\dagger \hat{a} \rangle^2}$ | Yes ($g^{(2)} \ge 1$) | No | Anti-bunching witness |
| **Squeezing Parameter** | $\langle (\Delta X_1)^2 \rangle$ | Yes ($\ge 1/4$) | Yes | Continuous-variable entanglement |
| **Wigner Negativity $\delta$** | $2 \iint_{W<0} |W| \, dx \, dp$ | **Zero ($\delta \equiv 0$)** | **Yes** | **CV Quantum Advantage Resource** |

---

### Key Takeaways
1. **Hudson's Theorem:** Pure Gaussian states are the unique pure states possessing positive Wigner functions ($W \ge 0$). All non-Gaussian pure states exhibit Wigner negativity.
2. **Negativity Volume $\delta(\hat{\rho})$:** Quantifies quantum non-classicality as the total absolute volume of $W(x, p) < 0$. It is faithfully zero for Gaussian states and strictly positive for non-Gaussian states.
3. **Quantum Computational Necessity:** Wigner negativity is a non-negotiable prerequisite for quantum speedup in continuous-variable architectures (Mari-Eisert theorem).

---

### Primary References
- **Hudson, R. L.** (1974). *When is the Wigner quasi-probability density non-negative?*. Reports on Mathematical Physics, 6(2), 249–252.
- **Kenfack, A., & Życzkowski, K.** (2004). *Negativity of the Wigner function as an indicator of non-classicality*. Journal of Optics B: Quantum and Semiclassical Optics, 6(10), 396–404.
- **Mari, A., & Eisert, J.** (2012). *Positive Wigner functions render classical simulation of quantum computation efficient*. Physical Review Letters, 109(23), 230503.
- **Weedbrook, C., et al.** (2012). *Gaussian quantum information*. Reviews of Modern Physics, 84(2), 621–669.

---

### Next in the Curriculum
In **Notebook 2.3: The Husimi Q-Function and Heterodyne Measurement**, we explore the smoothed phase-space distribution $Q(\alpha) = \frac{1}{\pi}\langle \alpha|\hat{\rho}|\alpha\rangle$, proving why it is everywhere non-negative ($Q \ge 0$) and corresponds to simultaneous noisy measurement of non-commuting quadratures.
